# Drone Detection Model Comparison

This notebook compares the performance of YOLOv8, YOLOv9, and EfficientDet on the drone detection dataset.

## 1. Setup and Install

In [ ]:
%pip install -q ultralytics effdet timm pycocotools albumentations torchmetrics

In [ ]:
import torch
import cv2
import numpy as np
import os
from ultralytics import YOLO
from effdet import create_model, DetBenchPredict
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from google.colab import drive
import zipfile

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Dataset Preparation
Mount Google Drive, unzip the dataset, and ensure data.yaml exists.

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Path to the zip file in Drive (UPDATE THIS IF NEEDED)
ZIP_PATH = '/content/drive/MyDrive/drone_dataset_split.zip'
EXTRACT_PATH = '/content/'
DATASET_DIR = os.path.join(EXTRACT_PATH, 'drone_dataset_split')

# Check if already extracted
if not os.path.exists(DATASET_DIR):
    print("Unzipping dataset...")
    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_PATH)
        print("Dataset unzipped!")
    else:
        print(f"Error: Zip file not found at {ZIP_PATH}. Please check the path.")
else:
    print("Dataset already exists.")

# Generate data.yaml (Fix for missing file)
yaml_content = f"""path: {DATASET_DIR}
train: images/train
val: images/val
test: images/test

names:
  0: drone
"""

yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
if not os.path.exists(yaml_path):
    print(f"Creating data.yaml at {yaml_path}...")
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    print("data.yaml created!")
else:
    print(f"data.yaml already exists at {yaml_path}")

## 3. Configuration
Update these paths to point to your trained model weights and dataset.

In [ ]:
# Model Weights Paths (UPDATE THESE!)
YOLOV8_WEIGHTS = '/content/drive/MyDrive/yolov8_project/runs/detect/train/weights/best.pt'
YOLOV9_WEIGHTS = '/content/drive/MyDrive/yolov9_project/runs/detect/train/weights/best.pt'
EFFDET_WEIGHTS = '/content/checkpoints/best_model.pth'

# Dataset Paths
# These assume the dataset was unzipped to /content/drone_dataset_split
# For EfficientDet (needs raw files)
VAL_IMAGES_DIR = '/content/drone_dataset_split/images/val'
VAL_LABELS_DIR = '/content/drone_dataset_split/labels/val'

# For YOLO (needs data.yaml)
YAML_DATA_PATH = '/content/drone_dataset_split/data.yaml'

## 4. EfficientDet Evaluation Logic
Since EfficientDet doesn't have a built-in one-liner for mAP, we define the dataset and evaluation loop here.

In [ ]:
class DroneDataset(Dataset):
    def __init__(self, images_dir, labels_dir, image_size=512, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_size = image_size
        self.transform = transform
        self.image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        self.image_files.sort()

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Resize info for restoring boxes if needed (simplified here to square resize)
        orig_h, orig_w = image.shape[:2]
        image = cv2.resize(image, (self.image_size, self.image_size))

        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(self.labels_dir, label_name)
        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_c, y_c, w, h = map(float, parts[1:5])
                        x_min = (x_c - w/2) * self.image_size
                        y_min = (y_c - h/2) * self.image_size
                        x_max = (x_c + w/2) * self.image_size
                        y_max = (y_c + h/2) * self.image_size
                        boxes.append([x_min, y_min, x_max, y_max])
                        labels.append(class_id)
        
        boxes = np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)
        labels = np.array(labels, dtype=np.int64) if labels else np.zeros((0,), dtype=np.int64)

        if self.transform:
            transformed = self.transform(image=image, bboxes=boxes, class_labels=labels)
            image = transformed['image']
            boxes = np.array(transformed['bboxes'], dtype=np.float32) if len(transformed['bboxes']) > 0 else np.zeros((0, 4), dtype=np.float32)
            labels = np.array(transformed['class_labels'], dtype=np.int64)
        
        # Normalize execution image
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        # Format for EffDet: [ymin, xmin, ymax, xmax]
        effdet_boxes = boxes.copy()
        if len(effdet_boxes) > 0:
            effdet_boxes = effdet_boxes[:, [1, 0, 3, 2]]

        # Return dictionary for testing
        target = {
            'boxes': torch.as_tensor(boxes, dtype=torch.float32), # PASS PASCAL VOC BOXES TO METRIC, NOT EFFDET FORMAT
            'effdet_boxes': torch.as_tensor(effdet_boxes, dtype=torch.float32),
            'labels': torch.as_tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([idx])
        }
        return image, target


In [ ]:
def evaluate_efficientdet(model_path, images_dir, labels_dir, device):
    if not os.path.exists(model_path):
        print(f"EfficientDet model not found at {model_path}. Skipping.")
        return None

    # Load model
    model = create_model('tf_efficientdet_d0', pretrained=False, num_classes=1, image_size=(512, 512))
    checkpoint = torch.load(model_path, map_location=device)
    # Handle state dict
    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    else:
        state_dict = checkpoint
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    bench = DetBenchPredict(model)
    bench.eval()

    # Dataset
    val_transform = A.Compose([
        A.ToFloat(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))
    
    dataset = DroneDataset(images_dir, labels_dir, transform=None) # No albumentations for simple inference conversion loop
    loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))

    metric = MeanAveragePrecision()
    
    print("Evaluating EfficientDet...")
    for images, targets in tqdm(loader):
        # Prepare images for EffDet
        # images is tuple of tensors. Stack them.
        # Resize is handled in dataset, but we need to ensure batching works
        img_batch = torch.stack(images).to(device)

        with torch.no_grad():
            # Output: [batch, max_det, 6] -> [x_min, y_min, x_max, y_max, score, class]
            output = bench(img_batch)

        # Update metric
        preds = []
        ground_truths = []
        
        for i in range(len(images)):
            # Predictions
            det = output[i]
            # Filter by confidence if needed, but mAP handles it
            pred_boxes = det[:, 0:4]
            scores = det[:, 4]
            labels = det[:, 5].int() - 1 # EffDet outputs 1-based classes often, dataset was 0-based. Check training!
            # Actually effdet training usually adds +1 to label. So predictions are 1-based. Metric expects 0-based matching target.
            # Let's assume standard behavior:
            
            # EffDet output is [xmin, ymin, xmax, ymax] (Pascal VOC)
            
            preds.append(dict(
                boxes=pred_boxes,
                scores=scores,
                labels=labels
            ))
            
            # Targets
            gt_boxes = targets[i]['boxes'].to(device)
            gt_labels = targets[i]['labels'].to(device)
            ground_truths.append(dict(
                boxes=gt_boxes,
                labels=gt_labels
            ))
        
        metric.update(preds, ground_truths)

    return metric.compute()


## 5. Run Everything and Compare

In [ ]:
results = {}

# 1. EfficientDet
print("--- Processing EfficientDet ---")
eff_metrics = evaluate_efficientdet(EFFDET_WEIGHTS, VAL_IMAGES_DIR, VAL_LABELS_DIR, device)
if eff_metrics:
    results['EfficientDet'] = {
        'mAP50': eff_metrics['map_50'].item(),
        'mAP50-95': eff_metrics['map'].item()
    }
    print(f"EfficientDet: mAP50={eff_metrics['map_50']:.4f}, mAP50-95={eff_metrics['map']:.4f}")
else:
    results['EfficientDet'] = {'mAP50': 0, 'mAP50-95': 0}

# 2. YOLOv8
print("\n--- Processing YOLOv8 ---")
if os.path.exists(YOLOV8_WEIGHTS):
    model_v8 = YOLO(YOLOV8_WEIGHTS)
    metrics_v8 = model_v8.val(data=YAML_DATA_PATH, split='test')
    results['YOLOv8'] = {
        'mAP50': metrics_v8.box.map50,
        'mAP50-95': metrics_v8.box.map
    }
else:
    print(f"YOLOv8 weights not found at {YOLOV8_WEIGHTS}")
    results['YOLOv8'] = {'mAP50': 0, 'mAP50-95': 0}

# 3. YOLOv9
print("\n--- Processing YOLOv9 ---")
if os.path.exists(YOLOV9_WEIGHTS):
    model_v9 = YOLO(YOLOV9_WEIGHTS)
    metrics_v9 = model_v9.val(data=YAML_DATA_PATH, split='test')
    results['YOLOv9'] = {
        'mAP50': metrics_v9.box.map50,
        'mAP50-95': metrics_v9.box.map
    }
else:
    print(f"YOLOv9 weights not found at {YOLOV9_WEIGHTS}")
    results['YOLOv9'] = {'mAP50': 0, 'mAP50-95': 0}

# Print Final Comparison
print("\n================ SUMMARY ================")
print(f"{'Model':<15} | {'mAP@50':<10} | {'mAP@50-95':<10}")
print("-"*41)
for model, res in results.items():
    print(f"{model:<15} | {res['mAP50']:.4f}     | {res['mAP50-95']:.4f}")